# 08 · Memory: Conversations 🧠

Every `responses.create` starts with a blank mind — unless we tell OpenAI which chat to continue.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
uncle = "You are a friendly kopitiam uncle. Keep replies short."

## 1. Without memory 🐟

In [ ]:
client.responses.create(model=MODEL, instructions=uncle, input="Hi Uncle, my name is Ahmad and I always drink teh-c.")

response = client.responses.create(model=MODEL, instructions=uncle, input="What's my name and my usual drink?")
print(response.output_text)

Uncle forgot already!

## 2. Memory way #1: `previous_response_id`
You used this in notebook 06. It says *"continue from this response"*.

In [ ]:
first = client.responses.create(model=MODEL, instructions=uncle, input="Hi Uncle, my name is Ahmad and I always drink teh-c.")

second = client.responses.create(
    model=MODEL, instructions=uncle, previous_response_id=first.id, input="What's my name and my usual drink?"
)
print(second.output_text)

It works, but we must keep track of the **latest** response ID after every message. If the bot restarts, the ID is lost.

## 3. Memory way #2: a conversation 💾
A **conversation** is a chat history that OpenAI keeps for us. It has **one ID that never changes** — easy to save!

In [ ]:
conversation = client.conversations.create()

print(conversation.id)

In [ ]:
client.responses.create(
    model=MODEL, instructions=uncle, conversation=conversation.id,
    input="Hi Uncle, my name is Ahmad and I always drink teh-c.",
)

response = client.responses.create(
    model=MODEL, instructions=uncle, conversation=conversation.id,
    input="What's my name and my usual drink?",
)
print(response.output_text)

## 4. Peek inside the conversation

In [ ]:
items = client.conversations.items.list(conversation.id, order="asc")

for item in items.data:
    if item.type == "message":
        print(f"{item.role:>9}: {item.content[0].text}")

## 5. Save the ID so Uncle never forgets
Your bot (`main.py`) reads `OPENAI_CONVERSATION_ID` from your `.env` file. Let's add it:

In [ ]:
with open(".env", "a") as f:
    f.write(f"\nOPENAI_CONVERSATION_ID={conversation.id}\n")

print("Saved! Open .env to check.")

> 👤 We're keeping it simple: **one bot, one person**. Everyone who chats with your bot shares this one conversation.

## ✏️ Exercises
1. Restart the kernel (⟳), run the setup cell, then chat using `conversation=os.environ["OPENAI_CONVERSATION_ID"]`. Does Uncle remember Ahmad?
2. Count how many messages are in your conversation.
3. Start fresh: create a new conversation and save its ID instead.

In [ ]:
# your code here